# Módulo 07 · Notebook 1: Pipelines, validación cruzada y GridSearch (Iris)

En el módulo 06 sufrimos con el preprocesamiento manual: ajustar el encoder solo en train, acordarse de transformar validación y test, y no poder reajustarlo dentro de la validación cruzada…

Un **`Pipeline`** encadena todos los pasos en **un solo objeto** que se comporta como un modelo:

```
  Pipeline = [ StandardScaler ] → [ PCA ] → [ LogisticRegression ]
                    │                │               │
  .fit(X, y)    fit+transform   fit+transform       fit
  .predict(X)     transform       transform        predict
```

> Basado en `sesion_7_pipelines.ipynb` (profesor Oscar N. Gómez). Cargamos Iris desde scikit-learn en lugar de descargarlo con `wget`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")
SEMILLA = 42

## 1. Los datos: Iris

150 flores de 3 especies, con 4 medidas (cm) de sépalo y pétalo. Es el "hola mundo" de la clasificación.

In [ ]:
iris = load_iris(as_frame=True)
data = iris.frame.rename(
    columns={
        "sepal length (cm)": "sepal_length_cm",
        "sepal width (cm)": "sepal_width_cm",
        "petal length (cm)": "petal_length_cm",
        "petal width (cm)": "petal_width_cm",
    }
)
data["species"] = data["target"].map(dict(enumerate(iris.target_names)))
data.head()

In [ ]:
sns.pairplot(data.drop(columns="target"), hue="species", height=2)
plt.show()

Los pétalos separan muy bien las especies; `setosa` se distingue fácilmente de las otras dos.

In [ ]:
X = data[["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]]
y = data["target"]  # 0 = setosa, 1 = versicolor, 2 = virginica

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEMILLA, stratify=y)
print(X_train.shape, X_test.shape)

## 2. PCA en un minuto

**PCA** (análisis de componentes principales) combina las variables originales en nuevas "direcciones" ordenadas por cuánta variabilidad explican.
Sirve para **reducir dimensiones** conservando la mayor parte de la información.

In [ ]:
pca = PCA().fit(StandardScaler().fit_transform(X_train))
varianza = pd.DataFrame(
    {"varianza_explicada": pca.explained_variance_ratio_, "acumulada": np.cumsum(pca.explained_variance_ratio_)},
    index=[f"PC{i + 1}" for i in range(4)],
).round(3)
varianza

Con **2 componentes** conservamos ~96 % de la información de las 4 variables originales.

## 3. El primer pipeline

In [ ]:
pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=2)),
        ("clf", LogisticRegression(max_iter=1000)),
    ]
)
pipeline

Cada paso tiene un **nombre** (`"scaler"`, `"pca"`, `"clf"`). Los nombres nos permiten cambiar pasos y buscar hiperparámetros más adelante.

## 4. Validación cruzada con el pipeline

Lo importante: **en cada fold el `StandardScaler` y el PCA se ajustan solo con los datos de entrenamiento de ese fold**. Cero *leakage*, sin escribir nada extra.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
pd.DataFrame(scores).round(3)

In [ ]:
print(f"Accuracy CV: {scores['test_score'].mean():.3f} ± {scores['test_score'].std():.3f}")

## 5. Cambiar el modelo con `set_params`

Como el pipeline es un objeto, podemos **cambiar solo el clasificador** y comparar varios con el mismo preprocesamiento:

In [ ]:
clasificadores = {
    "Logística": LogisticRegression(max_iter=1000),
    "SVC": SVC(),
    "KNN (k=3)": KNeighborsClassifier(n_neighbors=3),
    "Árbol": DecisionTreeClassifier(random_state=SEMILLA),
    "Random Forest": RandomForestClassifier(random_state=SEMILLA),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEMILLA),
}

filas = []
for nombre, clf in clasificadores.items():
    pipeline.set_params(clf=clf)
    res = cross_validate(pipeline, X_train, y_train, cv=cv, return_train_score=True)
    filas.append(
        {
            "modelo": nombre,
            "accuracy_train": res["train_score"].mean(),
            "accuracy_cv": res["test_score"].mean(),
            "std_cv": res["test_score"].std(),
            "tiempo_fit_s": res["fit_time"].mean(),
        }
    )
pd.DataFrame(filas).set_index("modelo").sort_values("accuracy_cv", ascending=False).round(3)

Árbol, Random Forest y Gradient Boosting tienen accuracy de train = **1.0** (memorizan el train), pero en CV bajan a 0.89-0.93.
Con PCA de 2 componentes perdimos algo de información y todos los modelos rondan el 90 %. ¿Podremos mejorar ajustando hiperparámetros?

## 6. Búsqueda de hiperparámetros con `GridSearchCV`

Los **hiperparámetros** son las "perillas" que **tú** fijas antes de entrenar (profundidad del árbol, número de árboles…). `GridSearchCV` prueba **todas las combinaciones** con validación cruzada.

Para llegar a un parámetro de un paso se usa **`<nombre_del_paso>__<parámetro>`** (con **dos** guiones bajos):

In [ ]:
pipeline.set_params(clf=GradientBoostingClassifier(random_state=SEMILLA))

grilla = {
    "pca__n_components": [2, 3, 4],
    "clf__n_estimators": [10, 50, 100],
    "clf__max_depth": [1, 2, 3],
}
n_combinaciones = np.prod([len(v) for v in grilla.values()])
print(f"{n_combinaciones} combinaciones × 5 folds = {n_combinaciones * 5} entrenamientos")

busqueda = GridSearchCV(pipeline, param_grid=grilla, cv=cv, scoring="accuracy", n_jobs=-1)
busqueda.fit(X_train, y_train)

print("Mejores hiperparámetros:", busqueda.best_params_)
print(f"Mejor accuracy CV: {busqueda.best_score_:.3f}")

In [ ]:
resultados_grid = pd.DataFrame(busqueda.cv_results_)
columnas = ["param_pca__n_components", "param_clf__n_estimators", "param_clf__max_depth", "mean_test_score", "rank_test_score"]
resultados_grid[columnas].sort_values("rank_test_score").head(8)

## 7. Evaluación final en test

`busqueda.best_estimator_` es el **pipeline completo**, ya reentrenado con todo el train usando los mejores hiperparámetros.

In [ ]:
y_pred = busqueda.predict(X_test)
print(f"Accuracy en test: {accuracy_score(y_test, y_pred):.3f}")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=iris.target_names, cmap="Greens")
plt.title("Mejor pipeline en test")
plt.show()

Las confusiones están entre `versicolor` y `virginica`, las dos especies que se solapaban en el *pairplot*. El EDA ya nos lo anticipaba.

¿Por qué la accuracy de test (~0.91) es menor que la de CV (~0.95)? El test tiene solo **45 flores**: cada error resta 2.2 puntos. Con datasets pequeños, las métricas "saltan" bastante; por eso reportamos la CV con su desviación estándar.

---
## Resumen

| Herramienta | Para qué |
|---|---|
| `Pipeline` | Encadenar preprocesamiento + modelo en un objeto |
| `cross_validate` | Evaluar con k folds, sin *leakage* |
| `set_params(clf=...)` | Cambiar un paso del pipeline |
| `GridSearchCV` | Probar combinaciones de hiperparámetros (`paso__parametro`) |
| `best_estimator_` | El pipeline ganador, listo para predecir o guardar |

## Ejercicios guiados (15 min)

1. Agrega `"SVC"` a una búsqueda con `clf__C: [0.1, 1, 10]` y `clf__kernel: ["linear", "rbf"]`. ¿Le gana al Gradient Boosting?
2. Quita el paso `"pca"` del pipeline. ¿Cambia la accuracy? ¿Por qué podría no hacer falta en Iris?
3. Usa `RandomizedSearchCV` con `n_iter=10` sobre la misma grilla del Gradient Boosting. ¿Cuánto tiempo ahorra? ¿Encuentra algo igual de bueno?

Soluciones en `ejercicios/soluciones/solucion_07_iris.py`.

In [ ]:
# Tu código aquí